# IOA-governed OpenAI Agents SDK example

Before this notebook's agent makes a real model call, it asks IOA's governance-service whether that call is allowed — the same `POST /enforce` decision point every other adapter in the platform uses (Kong's MCP proxy plugin, the n8n enforcement-client, ai-inference-service's own direct-request gateway).

**One-time setup, before running this notebook:**
1. `pip install -r requirements.txt`
2. Copy `.env.admin.example` to `.env.admin`, fill in an admin-scoped `IOA_ADMIN_API_KEY`.
3. Run `python setup_agent.py --agent-name "my-notebook-agent" --model-name gpt-4o --model-version 2026-08-01 --provider openai`
4. Copy `.env.example` to `.env`, paste in the values that command prints, plus your real `OPENAI_API_KEY`.

The credential in `.env` is a 15-minute delegation JWT (`auth.agents`' fixed TTL) — if it expires mid-session, re-run step 3 with `--credential-only` and update `.env`.

In [ ]:
from agents import Agent, Runner

from ioa_adapter import enforce_ai_model, token_seconds_remaining, IoaEnforcementError
import os

remaining = token_seconds_remaining()
if remaining is not None:
    print(f"IOA credential valid for {remaining}s")

MODEL_ID = os.environ["IOA_MODEL_ID"]  # the ai_inference.models row id from setup_agent.py

## The governed call

`enforce_ai_model` is called **before** `Runner.run_sync` — the enforcement check happens ahead of the real action, per the adapter contract ("run late relative to auth, early relative to the real action"). A `block` decision raises `IoaEnforcementError` and the agent never runs; a `warn` decision prints the warning and lets it proceed; `allow` is silent.

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You are a concise, helpful assistant.",
    model="gpt-4o",
)

def governed_run(prompt: str, *, data_classification: str | None = None) -> str:
    """Wraps Runner.run_sync with an IOA /enforce check first — this function,
    not Runner.run_sync directly, is the call site the rest of the notebook
    should use."""
    try:
        enforce_ai_model(MODEL_ID, data_classification=data_classification)
    except IoaEnforcementError as err:
        print(f"✗ blocked: {err}")
        raise

    result = Runner.run_sync(agent, prompt)
    return result.final_output

print(governed_run("Write a haiku about recursion."))

## What this does and doesn't cover

- **Covers**: gating this agent's own direct OpenAI model calls against whatever `ai_model` governance policy is seeded (sanctioned-provider checks, data-classification limits — see `AI Model Direct-Request Governance` in the Governance → Policies UI). Every call is delegation-tagged back to this notebook's registered agent identity, so `GET /api/auth/delegations` shows exactly which agent authorized which model call.
- **Doesn't cover**: governing individual *tools* this agent might call (that's `mcp_server`-asset governance, a separate integration — see `docs/enforcement-adapter-contract.md` if you wire MCP tools into this agent later), and doesn't do any static discovery of this notebook's own code the way `/ioa`'s workflow pipeline does for n8n workflows. This is registration + a runtime gate, not a governance QA scan of the notebook itself.